# Regional Sales Comparison

Goal: compare regional sales performance across revenue, order count, average order value, category mix, and growth trends.

This notebook is designed to answer five questions:
- Which regions lead on revenue?
- Which regions have the deepest order base?
- Which regions have the largest baskets?
- How does category mix differ by region?
- Which regions are strongest or weakest once scale and momentum are considered together?

In [ ]:
import warnings
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})

relative_path = pathlib.Path("Time Series Analysis") / "Time Series Forecasting" / "Sample - Superstore.xls"
search_roots = [pathlib.Path.cwd().resolve()] + list(pathlib.Path.cwd().resolve().parents)
candidate_paths = [root / relative_path for root in search_roots]
XLS_PATH = next((path for path in candidate_paths if path.exists()), None)

if XLS_PATH is None:
    raise FileNotFoundError(f"Could not locate dataset at {relative_path}")

df = pd.read_excel(XLS_PATH, sheet_name="Orders", engine="xlrd")
df.columns = df.columns.str.strip()
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Order Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Year"] = df["Order Date"].dt.year
df["Margin Pct"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"] * 100, np.nan)

print(f"Dataset path: {XLS_PATH}")
print(f"Rows loaded : {len(df):,}")
print(f"Date range  : {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")
print(f"Regions     : {', '.join(sorted(df['Region'].unique()))}")

In [ ]:
expected_columns = {"Order ID", "Order Date", "Region", "Category", "Sales", "Quantity", "Profit", "Discount", "Segment", "State", "Customer ID"}
missing_columns = sorted(expected_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

order_level = (
    df.groupby(["Region", "Order ID", "Order Date"], as_index=False)
      .agg(Order_Sales=("Sales", "sum"), Units=("Quantity", "sum"))
)

quality_check = pd.DataFrame({
    "Metric": ["Line items", "Distinct orders", "Distinct customers", "Regions", "Missing sales values", "Missing region values"],
    "Value": [len(df), df["Order ID"].nunique(), df["Customer ID"].nunique(), df["Region"].nunique(), int(df["Sales"].isna().sum()), int(df["Region"].isna().sum())],
})

region_summary = (
    df.groupby("Region")
      .agg(
          Revenue=("Sales", "sum"),
          Order_Count=("Order ID", "nunique"),
          Customers=("Customer ID", "nunique"),
          Units=("Quantity", "sum"),
          Profit=("Profit", "sum"),
          Avg_Discount=("Discount", "mean"),
      )
)
region_summary["Average Order Value"] = order_level.groupby("Region")["Order_Sales"].mean()
region_summary["Units per Order"] = order_level.groupby("Region")["Units"].mean()
region_summary["Margin %"] = np.where(region_summary["Revenue"] != 0, region_summary["Profit"] / region_summary["Revenue"] * 100, np.nan)
region_summary["Revenue Share %"] = region_summary["Revenue"] / region_summary["Revenue"].sum() * 100
region_summary["Order Share %"] = region_summary["Order_Count"] / region_summary["Order_Count"].sum() * 100
region_summary = region_summary.sort_values("Revenue", ascending=False)

display(quality_check)
display(region_summary.round({
    "Revenue": 0,
    "Order_Count": 0,
    "Customers": 0,
    "Units": 0,
    "Profit": 0,
    "Avg_Discount": 3,
    "Average Order Value": 2,
    "Units per Order": 2,
    "Margin %": 1,
    "Revenue Share %": 1,
    "Order Share %": 1,
}))

In [ ]:
plot_df = region_summary.reset_index()
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].bar(plot_df["Region"], plot_df["Revenue"] / 1e6, color=sns.color_palette("Blues_d", len(plot_df)))
axes[0].set_title("Revenue by Region")
axes[0].set_ylabel("Revenue ($M)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda value, _: f"${value:.1f}M"))

axes[1].bar(plot_df["Region"], plot_df["Order_Count"], color=sns.color_palette("Greens_d", len(plot_df)))
axes[1].set_title("Order Count by Region")
axes[1].set_ylabel("Distinct Orders")

axes[2].bar(plot_df["Region"], plot_df["Average Order Value"], color=sns.color_palette("Oranges_d", len(plot_df)))
axes[2].set_title("Average Order Value by Region")
axes[2].set_ylabel("AOV ($)")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda value, _: f"${value:.0f}"))

for ax in axes:
    ax.set_xlabel("Region")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

overall_aov = order_level["Order_Sales"].mean()
share_df = region_summary[["Revenue Share %", "Order Share %", "Average Order Value"]].copy()
share_df["AOV Index"] = share_df["Average Order Value"] / overall_aov * 100
display(share_df.round(1))

In [ ]:
monthly_region = (
    df.groupby(["Order Month", "Region"], as_index=False)
      .agg(Revenue=("Sales", "sum"))
)
monthly_region = monthly_region.sort_values(["Region", "Order Month"]).copy()
monthly_region["Rolling 3M Revenue"] = monthly_region.groupby("Region")["Revenue"].transform(lambda s: s.rolling(3, min_periods=1).mean())

fig, ax = plt.subplots(figsize=(12, 6))
for region in sorted(monthly_region["Region"].unique()):
    subset = monthly_region[monthly_region["Region"] == region]
    ax.plot(subset["Order Month"], subset["Rolling 3M Revenue"] / 1000, linewidth=2, label=region)

ax.set_title("Rolling 3-Month Revenue Trend by Region")
ax.set_xlabel("Order Month")
ax.set_ylabel("Revenue ($K)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda value, _: f"${value:,.0f}K"))
ax.legend(title="Region")
plt.tight_layout()
plt.show()

annual_revenue = df.pivot_table(values="Sales", index="Year", columns="Region", aggfunc="sum").sort_index()
annual_growth = annual_revenue.pct_change().mul(100)
periods = max(len(annual_revenue.index) - 1, 1)
growth_summary = pd.DataFrame({
    f"{int(annual_revenue.index.min())} Revenue": annual_revenue.iloc[0],
    f"{int(annual_revenue.index.max())} Revenue": annual_revenue.iloc[-1],
    "CAGR %": (((annual_revenue.iloc[-1] / annual_revenue.iloc[0]) ** (1 / periods)) - 1).mul(100).round(1),
    "Latest YoY %": annual_growth.iloc[-1].round(1),
})

display(growth_summary.sort_values("CAGR %", ascending=False))

In [ ]:
category_mix = df.pivot_table(values="Sales", index="Region", columns="Category", aggfunc="sum", fill_value=0)
category_share = category_mix.div(category_mix.sum(axis=1), axis=0).mul(100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(category_share, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.5, ax=ax, cbar_kws={"label": "Revenue share %"})
ax.set_title("Category Revenue Share by Region")
ax.set_xlabel("Category")
ax.set_ylabel("Region")
plt.tight_layout()
plt.show()

segment_share = df.pivot_table(values="Sales", index="Region", columns="Segment", aggfunc="sum", fill_value=0)
segment_share = segment_share.div(segment_share.sum(axis=1), axis=0).mul(100)
state_share = df.pivot_table(values="Sales", index="Region", columns="State", aggfunc="sum", fill_value=0)
state_share = state_share.div(state_share.sum(axis=1), axis=0).mul(100)

driver_table = pd.DataFrame({
    "Revenue": region_summary["Revenue"],
    "Order Count": region_summary["Order_Count"],
    "Average Order Value": region_summary["Average Order Value"],
    "Margin %": region_summary["Margin %"],
    "Top Category": category_share.idxmax(axis=1),
    "Top Category Share %": category_share.max(axis=1),
    "Top Segment": segment_share.idxmax(axis=1),
    "Top Segment Share %": segment_share.max(axis=1).round(1),
    "Top State": state_share.idxmax(axis=1),
    "Top State Share %": state_share.max(axis=1).round(1),
}).join(growth_summary[["CAGR %", "Latest YoY %"]])

display(driver_table.round({
    "Revenue": 0,
    "Order Count": 0,
    "Average Order Value": 2,
    "Margin %": 1,
    "Top Category Share %": 1,
    "Top Segment Share %": 1,
    "Top State Share %": 1,
    "CAGR %": 1,
    "Latest YoY %": 1,
}))

In [ ]:
ranking = driver_table.copy()
ranking["Revenue Rank"] = ranking["Revenue"].rank(ascending=False, method="min")
ranking["Order Rank"] = ranking["Order Count"].rank(ascending=False, method="min")
ranking["AOV Rank"] = ranking["Average Order Value"].rank(ascending=False, method="min")
ranking["Growth Rank"] = ranking["CAGR %"].rank(ascending=False, method="min")
ranking["Composite Rank"] = ranking[["Revenue Rank", "Order Rank", "AOV Rank", "Growth Rank"]].mean(axis=1)
ranking = ranking.sort_values(["Composite Rank", "Margin %"], ascending=[True, False])

best_region = ranking.index[0]
weakest_region = ranking.index[-1]
median_values = ranking[["Average Order Value", "CAGR %", "Margin %"]].median()

def describe_region(region_name):
    row = ranking.loc[region_name]
    notes = []
    notes.append("larger baskets" if row["Average Order Value"] >= median_values["Average Order Value"] else "smaller baskets")
    notes.append("above-median growth" if row["CAGR %"] >= median_values["CAGR %"] else "slower growth")
    notes.append("healthy margin" if row["Margin %"] >= median_values["Margin %"] else "margin pressure")
    notes.append(f"{row['Top Category']} leads the mix ({row['Top Category Share %']:.1f}% of sales)")
    notes.append(f"{row['Top Segment']} is the largest segment ({row['Top Segment Share %']:.1f}% of sales)")
    return "; ".join(notes)

display(ranking[["Revenue", "Order Count", "Average Order Value", "CAGR %", "Latest YoY %", "Margin %", "Composite Rank", "Top Category", "Top Segment"]].round({
    "Revenue": 0,
    "Order Count": 0,
    "Average Order Value": 2,
    "CAGR %": 1,
    "Latest YoY %": 1,
    "Margin %": 1,
    "Composite Rank": 2,
}))

print("=" * 72)
print("REGIONAL SALES COMPARISON - EXECUTIVE TAKEAWAY")
print("=" * 72)
print(f"Strongest overall region: {best_region}")
print(f"Weakest overall region : {weakest_region}")
print()
for region_name in [best_region, weakest_region]:
    row = ranking.loc[region_name]
    print(region_name)
    print(f"  Revenue             : ${row['Revenue']:,.0f}")
    print(f"  Order count         : {int(row['Order Count']):,}")
    print(f"  Average order value : ${row['Average Order Value']:.2f}")
    print(f"  CAGR                : {row['CAGR %']:.1f}%")
    print(f"  Margin              : {row['Margin %']:.1f}%")
    print(f"  Likely drivers      : {describe_region(region_name)}")
    print()

## Business Recommendations

1. Protect the strongest region by keeping its lead category in stock and preserving service levels for its dominant customer segment.
2. For the weakest region, fix assortment and basket size before using broad discounts. Mix problems are usually more expensive than volume problems.
3. If a region has healthy order count but weak average order value, bundle complementary products and upsell within the dominant category.
4. If a region is concentrated in one category or one state, diversify local assortment to reduce concentration risk.
5. Use the growth table to distinguish a temporary dip from a structural regional weakness before reallocating budget.

---
Notebook: Regional Sales Comparison
Dataset: Sample Superstore (Orders sheet)
Techniques: regional scorecards, growth analysis, category mix heatmaps, composite ranking